# CS6140 Machine Learning
## Problem Set 1

This problem set asks you to:

1. Review several mathematical properties that are central to linear algebra as applied to machine learning
2. Practice using `numpy` for mathematical coding in Python

**I suggest that you start this problem set by copying the problems on paper and thinking about how you would solve them by hand.** This should make it much easier to write the code.

The single most useful reference for this module is likely the [`numpy` User Guide](https://numpy.org/devdocs/user/index.html).

_This is an experimental assignment and I will award generous partial credit for answers demonstrating understanding of the concepts even if the code does not work perfectly._

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Callable

# Set random seed for reproducibility
np.random.seed(42)

## Problem 1: Matrix Multiplication as Sum of Outer Products (5 pts)

Verify that for matrices $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{n \times p}$, we can compute:

$$C = AB = \sum_{i=1}^{n} a_i b_i^T$$

where $a_i$ is the $i$-th column of $A$ and $b_i^T$ is the $i$-th column of $B$.

In [38]:
def matrix_multiply_outer_product(A, B):
    """
    Compute matrix multiplication C = AB using sum of outer products.
    
    Args:
        A: Matrix of shape (m, n)
        B: Matrix of shape (n, p)
    
    Returns:
        C: Matrix of shape (m, p)
    """
    m, n = A.shape
    p = B.shape[1]
    
    # Starting with a blank matrix
    C = np.zeros((m, p))
    
    for k in range(n):
        # Taking the k-th column and k-th row
        col_k = A[:, k]
        row_k = B[k, :]
        
        # Manually adding the outer product to our result
        C += np.outer(col_k, row_k)
    
    return C

In [44]:
def test_problem_1():
    """TODO: Write a test case and use np.allclose to verify that your implementation matches the result from np.dot."""
    m, n, p = 10, 5, 20
    A = np.random.rand(m, n)
    B = np.random.rand(n, p)
    C = matrix_multiply_outer_product(A, B)
    assert np.allclose(C, np.dot(A, B)), "Matrix multiplication result does not match np.dot"
    print("✓ Problem 1: Matrix multiplication test passed!")

test_problem_1()

✓ Problem 1: Matrix multiplication test passed!


## Problem 2: Symmetric and Anti-symmetric Decomposition (5 pts)

Show that any square matrix $A \in \mathbb{R}^{n \times n}$ can be written as $A = B + C$,
where $B$ is symmetric and $C$ is anti-symmetric (skew-symmetric).

Derive and implement the formulas for $B$ and $C$.

In [ ]:
def decompose_symmetric_antisymmetric(A):
    """
    Decompose a square matrix into symmetric and anti-symmetric components.
    
    Args:
        A: Square matrix of shape (n, n)
    
    Returns:
        B: Symmetric component (B = B^T)
        C: Anti-symmetric component (C = -C^T)
    """
    # Derivation:
    # We want to find B and C such that:
    # 1. $A = B + C$
    # 2. $B = B^T$ (Symmetric)
    # 3. $C = -C^T$ (Anti-symmetric)
    #
    # Taking the transpose of equation 1:
    # $A^T = (B + C)^T = B^T + C^T$
    # Substituting the properties from 2 and 3:
    # $A^T = B - C$
    #
    # Solving the system:
    # $(A + A^T) = (B + C) + (B - C) = 2B -> \implies B = \frac{1}{2}(A + A^T)$
    # $(A - A^T) = (B + C) - (B - C) = 2C -> \implies C = \frac{1}{2}(A - A^T)$
    B = (A + A.T) / 2
    C = (A - A.T) / 2
    return B, C

In [45]:
def test_problem_2():
    """TODO: Write a test case that uses np.allclose to verify that your implementation can reconstruct A."""
    n = 10
    A = np.random.randn(n, n)
    B, C = decompose_symmetric_antisymmetric(A)
    assert np.allclose(B, B.T), "B is not symmetric"
    assert np.allclose(C, -C.T), "C is not anti-symmetric"
    assert np.allclose(B + C, A), "Does not reconstruct A"
    print("✓ Problem 2: Decomposition test passed!")

test_problem_2()

✓ Problem 2: Decomposition test passed!


## Problem 3: Matrix Calculus (12 pts total, 4 pts each)

Verify the following derivatives numerically using finite differences:

**(a)** $\frac{\partial}{\partial x}(a^T x) = a$

**(b)** $\frac{\partial}{\partial x}(x^T A x) = (A + A^T)x$

**(c)** $\frac{\partial}{\partial x}(\|y - Ax\|_2^2) = 2A^T(Ax - y)$

In [42]:
def gradient_linear(x, a):
    """
    Compute the gradient of f(x) = a^T x with respect to x.

    """
    # Grad of a^T x is just a
    return a

def gradient_quadratic(x, A):
    """
    Compute the gradient of f(x) = x^T A x with respect to x.

    """
    # d/dx(ax^2) = 2ax. 
    # Here, we have 'x' on both sides of A, so it is like a product rule.
    # 1. Derivative of the left 'x' gives: A * x
    # 2. Derivative of the right 'x' gives: A.T * x
    # Total = (A + A.T) * x
    return (A + A.T) @ x

def gradient_least_squares(x, A, y):
    """
    Compute the gradient of f(x) = ||y - Ax||_2^2 with respect to x.

    """
    # Let u = (Ax - y).
    # We are finding the derivative of (u^2).
    #
    # 1. Derivative of the "outside" (the square) is 2 * u.
    # 2. Derivative of the "inside" (Ax - y) with respect to x is just A.
    # 
    # When we put them together in matrix form, we have to flip A 
    # to A-transpose so the shapes match up for multiplication.
    # Result: 2 * A.T * (Ax - y)
    residual = (A @ x) - y
    return 2 * A.T @ residual


def numerical_gradient(f, x, epsilon=1e-7):
    """
    Compute numerical gradient using finite differences.
    
    This is provided for you to test your analytical gradients.
    """
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[i] += epsilon
        x_minus[i] -= epsilon
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * epsilon)
    return grad

In [43]:
def test_problem_3():
    """Test gradient computations."""
    n = 5
    x = np.random.randn(n)
    a = np.random.randn(n)
    A = np.random.randn(n, n)
    y = np.random.randn(n)
    
    # Test 3a: Linear gradient
    analytical_grad_a = gradient_linear(x, a)
    numerical_grad_a = numerical_gradient(lambda x: a.T @ x, x)
    assert np.allclose(analytical_grad_a, numerical_grad_a, atol=1e-5), "3a: Gradient incorrect!"
    assert np.allclose(analytical_grad_a, a, atol=1e-5), "3a: Should equal a!"
    print("✓ Problem 3a: Linear gradient test passed!")
    
    # Test 3b: Quadratic gradient
    analytical_grad_b = gradient_quadratic(x, A)
    numerical_grad_b = numerical_gradient(lambda x: x.T @ A @ x, x)
    assert np.allclose(analytical_grad_b, numerical_grad_b, atol=1e-5), "3b: Gradient incorrect!"
    print("✓ Problem 3b: Quadratic gradient test passed!")
    
    # Test 3c: Least squares gradient
    analytical_grad_c = gradient_least_squares(x, A, y)
    numerical_grad_c = numerical_gradient(lambda x: np.sum((y - A @ x)**2), x)
    assert np.allclose(analytical_grad_c, numerical_grad_c, atol=1e-5), "3c: Gradient incorrect!"
    print("✓ Problem 3c: Least squares gradient test passed!")

test_problem_3()

✓ Problem 3a: Linear gradient test passed!
✓ Problem 3b: Quadratic gradient test passed!
✓ Problem 3c: Least squares gradient test passed!


## Problem 4: Positive Semidefinite Matrices (6 pts)

Verify that for any matrix $X \in \mathbb{R}^{m \times n}$, the Gram matrix $G = XX^T$
is always positive semidefinite.

A matrix is positive semidefinite if:
1. It's symmetric
2. All eigenvalues are non-negative
3. For all vectors $v$, $v^T G v \geq 0$

In [ ]:
def verify_positive_semidefinite(X, num_test_vectors=100):
    """
    Verify that G = XX^T is positive semidefinite.
    
    Args:
        X: Matrix of shape (m, n)
        num_test_vectors: Number of random vectors to test
    
    Returns:
        Dictionary with verification results
    
    """
    # 1. Compute G = XX^T
    G = X @ X.T
    
    # 2. for a matrix to be PSD, it needs to be symmetric first. Check 
    is_sym = np.allclose(G, G.T)
    
    # 3. Non-negative eigenvalues?
    # If any factor is negative, it means the matrix "flips" space
    # So, for PSD, all these factors must be >= 0.
    e_vals = np.linalg.eigvals(G)
    min_ev = np.min(e_vals)
    psd_ev = bool(min_ev > -1e-9)
    
    # 4. Test v^T G v ≥ 0 for multiple random vectors v
    q_tests = []
    for _ in range(num_test_vectors):
        v = np.random.randn(G.shape[0])
        score = v.T @ G @ v
        q_tests.append(score)
        
    return {
        'is_symmetric': is_sym,
        'min_eigenvalue': min_ev,
        'all_eigenvalues_non_negative': psd_ev,
        'quadratic_form_tests': q_tests
    }

In [31]:
def test_problem_4():
    """Test positive semidefinite property."""
    # Test various matrix shapes
    test_cases = [
        (3, 3),   # Square
        (5, 3),   # Tall
        (3, 5),   # Wide
        (10, 7),  # Larger
    ]
    
    for m, n in test_cases:
        X = np.random.randn(m, n)
        results = verify_positive_semidefinite(X)
        
        assert results['is_symmetric'], f"G not symmetric for shape ({m}, {n})!"
        assert results['all_eigenvalues_non_negative'], f"Negative eigenvalue found for shape ({m}, {n})!"
        assert all(qf >= -1e-10 for qf in results['quadratic_form_tests']), \
            f"Negative quadratic form for shape ({m}, {n})!"
        
        print(f"✓ Problem 4: Shape ({m}, {n}) - Min eigenvalue: {results['min_eigenvalue']:.2e}")
    
    print("✓ Problem 4: All tests passed!")

test_problem_4()

✓ Problem 4: Shape (3, 3) - Min eigenvalue: 9.04e-02
✓ Problem 4: Shape (5, 3) - Min eigenvalue: -1.29e-15
✓ Problem 4: Shape (3, 5) - Min eigenvalue: 2.79e-01
✓ Problem 4: Shape (10, 7) - Min eigenvalue: -1.48e-15
✓ Problem 4: All tests passed!


## Problem 5: Norm Properties (10 pts)

Verify that $\ell_1$ and $\ell_\infty$ norms satisfy the four properties of a norm:

1. Homogeneity: $f(sx) = |s|f(x)$
2. Triangle Inequality: $f(x + y) \leq f(x) + f(y)$
3. Positive Definiteness: $f(x) = 0 \Rightarrow x = 0$
4. Non-negativity: $f(x) \geq 0$

In [33]:
def verify_norm_properties(norm_func, norm_name, num_tests=100, n=10):
    """
    Verify that a given norm function satisfies all four norm properties.
    
    Args:
        norm_func: Function that computes the norm
        norm_name: Name of the norm for printing
        num_tests: Number of random test cases
        n: Dimension of test vectors
    
    Returns:
        Dictionary with test results
    
    TODO: Test all four properties:
    1. Homogeneity: Test with random scalars and vectors
    2. Triangle inequality: Test with random vector pairs
    3. Positive definiteness: Test that norm(0) = 0 and norm(x) > 0 for x ≠ 0
    4. Non-negativity: Test that all norms are ≥ 0
    """
    results = {
        'homogeneity_passed': [], 
        'triangle_inequality_passed': [], 
        'positive_definiteness_passed': True, 
        'non_negativity_passed': [] 
    }
    
    # First, check the Positive Definiteness: f(0) must be 0
    if not np.isclose(norm_func(np.zeros(n)), 0):
        results['positive_definiteness_passed'] = False

    # Single loop to run all randomized tests
    for _ in range(num_tests):
        # Generate random inputs for this round
        x = np.random.randn(n)
        y = np.random.randn(n)
        s = np.random.randn()
        
        # 1. Homogeneity
        # Checking if f(s*x) is the same as |s| * f(x)
        is_homo = np.isclose(norm_func(s * x), np.abs(s) * norm_func(x))
        results['homogeneity_passed'].append(is_homo)

        # 2. Triangle Inequality
        # Checking if going straight (x+y) is shorter than the two-step path
        is_tri = norm_func(x + y) <= (norm_func(x) + norm_func(y) + 1e-11)
        results['triangle_inequality_passed'].append(is_tri)
        
        # 4. Non-negativity
        # Checking if the result is always 0 or higher
        results['non_negativity_passed'].append(norm_func(x) >= -1e-11)
        
        # 3. Positive Definiteness (Part 2)
        # If the vector is not zero, the length should not be zero.
        # (Since x is random, it is essentially never exactly a zero vector)
        if np.isclose(norm_func(x), 0):
             results['positive_definiteness_passed'] = False
             
    return results

In [34]:
def test_problem_5():
    """Test norm properties for L1 and L-infinity norms."""
    
    # L1 norm
    l1_norm = lambda x: np.sum(np.abs(x))
    results_l1 = verify_norm_properties(l1_norm, "L1")
    
    assert all(results_l1['homogeneity_passed']), "L1: Homogeneity failed!"
    assert all(results_l1['triangle_inequality_passed']), "L1: Triangle inequality failed!"
    assert results_l1['positive_definiteness_passed'], "L1: Positive definiteness failed!"
    assert all(results_l1['non_negativity_passed']), "L1: Non-negativity failed!"
    print("✓ Problem 5: L1 norm - All properties verified!")
    
    # L-infinity norm
    linf_norm = lambda x: np.max(np.abs(x))
    results_linf = verify_norm_properties(linf_norm, "L∞")
    
    assert all(results_linf['homogeneity_passed']), "L∞: Homogeneity failed!"
    assert all(results_linf['triangle_inequality_passed']), "L∞: Triangle inequality failed!"
    assert results_linf['positive_definiteness_passed'], "L∞: Positive definiteness failed!"
    assert all(results_linf['non_negativity_passed']), "L∞: Non-negativity failed!"
    print("✓ Problem 5: L∞ norm - All properties verified!")

test_problem_5()

✓ Problem 5: L1 norm - All properties verified!
✓ Problem 5: L∞ norm - All properties verified!


## Problem 6: Convexity Verification (8 pts)

Determine whether the following functions are convex by checking if:

$$f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda)f(y) \text{ for all } \lambda \in [0, 1]$$

**(a)** $f(x) = e^{ax}$ for fixed $a \in \mathbb{R}$

**(b)** $f(x) = |x|^p$ for $p > 1$

**(c)** $f(x) = -\log(x)$ for $x \in (0, +\infty)$

**(d)** $f(\cdot)$ where $f$ is a norm

In [35]:
def check_convexity_numerical(f, x_range, num_tests=1000):
    """
    Check if a univariate function is convex numerically.
    
    Args:
        f: Function to test
        x_range: (min, max) range for testing
        num_tests: Number of random test points
    
    Returns:
        bool: True if the function appears convex
    
    TODO: Test convexity by:
    1. Sampling num_tests random pairs of points x, y in the x_range
    2. Sampling random λ ∈ [0, 1]
    3. Checking if f(λx + (1-λ)y) ≤ λf(x) + (1-λ)f(y)
    4. Return the results including any violations found
    """

    is_convex = True
    
    for _ in range(num_tests):
        # 1. Sample random pair of points in the range
        x = np.random.uniform(x_range[0], x_range[1])
        y = np.random.uniform(x_range[0], x_range[1])
        
        # 2. Sample random lambda between 0 and 1
        # This determines the "mix" or where we are on the line
        lam = np.random.uniform(0, 1)
        
        # 3. Checking Jensen's inequality: f(λx + (1-λ)y) ≤ λf(x) + (1-λ)f(y)
        val_at_mix = f(lam * x + (1 - lam) * y)
        line_height = lam * f(x) + (1 - lam) * f(y)
        
        # Using a tiny 1e-11 tolerance for floating point precision
        if val_at_mix > line_height + 1e-11:
            is_convex = False
            # found a violation
            break
            
    return is_convex

## Problem 7: Implement the test case for 6d (4 points)

In [36]:
def test_problem_6():
    """Test convexity of various functions."""
    
    print("\n" + "="*60)
    print("Problem 6: Convexity Tests")
    print("="*60)
    
    # 6a: f(x) = e^(ax)
    print("\n6a: Testing f(x) = e^(ax)")
    for a in [-2, 0, 2]:
        f_exp = lambda x: np.exp(a * x)
        result = check_convexity_numerical(f_exp, (-2, 2))
        print(f"  a = {a:2d}: Convex = {result}")
    
    # 6b: f(x) = |x|^p
    print("\n6b: Testing f(x) = |x|^p")
    for p in [1.5, 2.0, 3.0]:
        f_power = lambda x: np.abs(x) ** p
        result = check_convexity_numerical(f_power, (-2, 2))
        print(f"  p = {p:.1f}: Convex = {result}")
    
    # 6c: f(x) = -log(x)
    print("\n6c: Testing f(x) = -log(x)")
    f_neglog = lambda x: -np.log(x)
    result = check_convexity_numerical(f_neglog, (0.1, 10))
    print(f"  Convex = {result}")
    
    # 6d: Norms are convex
    print("\n6d: Testing norms in R^n")
    # Since norms deal with vectors, we test along a "line" in n-dimensional space.
    # We pick two random vectors 'u' and 'v' and check if the 'average vector' has a shorter length than the 'average of the lengths'.
    
    is_convex_norm = True
    n_dim = 5 # testing in 5D space
    for _ in range(1000):
        # 1. Pick two random vectors
        u = np.random.randn(n_dim)
        v = np.random.randn(n_dim)
        
        # 2. Pick a random mix percentage
        t = np.random.rand()
        
        # 3. Jensen's Inequality for vectors: ||tu + (1-t)v|| <= t||u|| + (1-t)||v||
        val_mixed_vector = np.linalg.norm(t * u + (1 - t) * v)
        val_mixed_heights = t * np.linalg.norm(u) + (1 - t) * np.linalg.norm(v)
        
        if val_mixed_vector > val_mixed_heights + 1e-11:
            is_convex_norm = False
            break
            
    print(f"  L2 Norm Convex = {is_convex_norm}")
    
    print("\n" + "="*60)

test_problem_6()


Problem 6: Convexity Tests

6a: Testing f(x) = e^(ax)
  a = -2: Convex = True
  a =  0: Convex = True
  a =  2: Convex = True

6b: Testing f(x) = |x|^p
  p = 1.5: Convex = True
  p = 2.0: Convex = True
  p = 3.0: Convex = True

6c: Testing f(x) = -log(x)
  Convex = True

6d: Testing norms in R^n
  L2 Norm Convex = True



## You made it!